In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.linear_model import  LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler ,MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [36]:
df=pd.read_csv('New Dataset/spain.csv')
df.head()


,Y,X,data_payload_id,instance_datetime,url,agency,platform_type,platform_id,platform_name,gaw_id,...,daily_utc_begin,daily_utc_end,daily_utc_mean,daily_nobs,daily_mmu,daily_columnso2,latest_observation,country,scientific_authority,version
0,28.46,-16.25,2306481,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.15,10.52,9.60,4.0,2.181,-0.5,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
1,28.46,-16.25,2306502,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.50,17.88,15.22,12.0,1.421,-1.4,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
2,28.46,-16.25,2306482,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,8.85,14.03,10.80,16.0,1.638,-0.9,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
3,28.46,-16.25,2306480,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.83,17.27,15.41,10.0,1.602,-0.6,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
4,28.46,-16.25,2306483,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.55,17.55,15.04,19.0,1.680,-0.8,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0


In [37]:
df=df.drop(['Y', 'X', 'data_payload_id', 'instance_datetime', 'url', 'agency',
       'platform_type', 'platform_id', 'platform_name', 'gaw_id',
       'instrument_name', 'instrument_model', 'instrument_number',
       'monthly_date', 'monthly_stddevo3', 'monthly_npts','daily_stddevo3',
        'daily_wlcode', 'daily_obscode', 'monthly_columno3',
        'daily_utc_begin', 'daily_utc_end', 'daily_utc_mean',
       'daily_nobs', 'daily_mmu', 'daily_columnso2', 'latest_observation',
       'country', 'scientific_authority', 'version'], axis = 1)
df.head()

,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [38]:
df=df.drop_duplicates()
df=df.dropna()
df.head()


,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [39]:
print(f"Date Range: {df.loc[:,'daily_date'][len(df)-1]} to {df.loc[:,'daily_date'][0]}")

Date Range: 2001-03-10 to 2025-03-06


**# Trial 1** <br>
***TimeSeriesSplit + lag_60 + rolling_avg_7 + exp_avg_7*** <br>


In [40]:
# 2. Sort the dataframe by the full date (oldest to latest)
df1=df.copy()
df1 = df1.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df1['rolling_avg'] = df1['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df1['ema_avg'] = df1['daily_columno3'].shift(1).ewm(span=7, adjust=False).mean()


for i in range(1, 61):
    df1[f'lag{i}'] = df1['daily_columno3'].shift(i)


df1.dropna(inplace=True)
print(df1.shape)
df1.head()

(65450, 64)


,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,290.176070,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,302.632053,340.0,286.0,287.4,297.2,311.8,294.2,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,303.974039,308.0,340.0,286.0,287.4,297.2,311.8,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,333.980530,424.0,308.0,340.0,286.0,287.4,297.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,331.235397,323.0,424.0,308.0,340.0,286.0,287.4,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [41]:
df1.tail()

,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
65505,2025-03-31,304.2,318.200000,321.528513,337.5,323.5,296.8,314.8,317.0,310.6,...,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6,403.9,309.3
65506,2025-03-31,326.3,314.914286,317.196385,304.2,337.5,323.5,296.8,314.8,317.0,...,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6,403.9
65507,2025-03-31,332.6,317.157143,319.472289,326.3,304.2,337.5,323.5,296.8,314.8,...,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6
65508,2025-03-31,291.9,319.385714,322.754217,332.6,326.3,304.2,337.5,323.5,296.8,...,308.4,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2
65509,2025-03-31,325.0,316.114286,315.040662,291.9,332.6,326.3,304.2,337.5,323.5,...,368.1,308.4,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6


In [42]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
lag_features.append('ema_avg')
X = df1[lag_features]
y = df1['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000791 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 10910, number of used features: 62
[LightGBM] [Info] Start training from score -0.074399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001260 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 21818, number of used features: 62
[LightGBM] [Info] Start training from score 0.004540


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002041 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 32726, number of used features: 62
[LightGBM] [Info] Start training from score 0.068249
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 43634, number of used features: 62
[LightGBM] [Info] Start training from score 0.028112


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002821 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 54542, number of used features: 62
[LightGBM] [Info] Start training from score -0.001818
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.758
MAE : 0.555
R2  : 42.57 %

----- Decision Tree -----
RMSE: 0.998
MAE : 0.736
R2  : 0.34 %

----- Random Forest -----
RMSE: 0.759
MAE : 0.558
R2  : 42.43 %

----- Gradient Boosting -----
RMSE: 0.757
MAE : 0.555
R2  : 42.68 %

----- Support Vector Regressor -----
RMSE: 0.772
MAE : 0.568
R2  : 40.38 %

----- XGBoost -----
RMSE: 0.781
MAE : 0.571
R2  : 39.07 %

----- LightGBM -----
RMSE: 0.758
MAE : 0.557
R2  : 42.49 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***# Trial 2 : TimeSeriesSplit + lag_30 + rolling_std_3 + exp_avg_3***

In [43]:
# 2. Sort the dataframe by the full date (oldest to latest)
df2=df.copy()
df2 = df2.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df2['rolling_std'] = df2['daily_columno3'].shift(1).rolling(window=3).std()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df2['ema_avg'] = df2['daily_columno3'].shift(1).ewm(span=3, adjust=False).mean()


for i in range(1, 31):
    df2[f'lag{i}'] = df2['daily_columno3'].shift(i)


df2.dropna(inplace=True)
print(df2.shape)
df2.head()

(65480, 34)


,daily_date,daily_columno3,rolling_std,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag21,lag22,lag23,lag24,lag25,lag26,lag27,lag28,lag29,lag30
30,1976-09-07,273.8,4.409460,287.181151,289.4,280.6,284.5,290.3,295.2,303.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
31,1976-09-08,284.5,7.821338,280.490575,273.8,289.4,280.6,284.5,290.3,295.2,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
32,1976-09-09,283.5,7.977677,282.495288,284.5,273.8,289.4,280.6,284.5,290.3,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
33,1976-09-10,283.5,5.910161,282.997644,283.5,284.5,273.8,289.4,280.6,284.5,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
34,1976-09-13,307.9,0.577350,283.248822,283.5,283.5,284.5,273.8,289.4,280.6,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [44]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 31)]
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df2[lag_features]
y = df2['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000511 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 10915, number of used features: 32
[LightGBM] [Info] Start training from score -0.074213
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000747 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 21828, number of used features: 32
[LightGBM] [Info] Start training from score 0.003913
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000908 seconds.
You can set `f

/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 43654, number of used features: 32
[LightGBM] [Info] Start training from score 0.027864
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 54567, number of used features: 32
[LightGBM] [Info] Start training from score -0.001854
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.755
MAE : 0.553
R2  : 42.98 %

----- Decision Tree -----
RMSE: 1.002
MAE : 0.736
R2  : -0.36 %

----- Random Forest -----
RMSE: 0.760
MAE : 0.558
R2  : 42.21 %

----- Gradient Boosting -----
RMSE: 0.759
MAE : 0.555
R2  : 42.45 %

----- Support Vector

/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 3 : TimeSeriesSplit + lag_80 + rolling_avg_14 + rolling_std_14 + exp_avg_14**

In [45]:
# 2. Sort the dataframe by the full date (oldest to latest)
df3=df.copy()
df3 = df3.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df3['rolling_std'] = df3['daily_columno3'].shift(1).rolling(window=14).std()
df3['rolling_avg'] = df3['daily_columno3'].shift(1).rolling(window=14).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df3['ema_avg'] = df3['daily_columno3'].shift(1).ewm(span=14, adjust=False).mean()


for i in range(1, 81):
    df3[f'lag{i}'] = df3['daily_columno3'].shift(i)


df3.dropna(inplace=True)
print(df3.shape)
df3.head()

(65430, 85)


,daily_date,daily_columno3,rolling_std,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,...,lag71,lag72,lag73,lag74,lag75,lag76,lag77,lag78,lag79,lag80
80,1980-04-28,373.0,21.702914,360.642857,356.398440,354.0,347.0,344.0,360.0,386.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
81,1980-04-30,413.0,21.225762,360.071429,358.611981,373.0,354.0,347.0,344.0,360.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
82,1980-05-12,389.0,23.968500,365.785714,365.863717,413.0,373.0,354.0,347.0,344.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
83,1980-05-13,373.0,24.464821,368.285714,368.948555,389.0,413.0,373.0,354.0,347.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
84,1980-05-14,376.0,21.699749,371.428571,369.488748,373.0,389.0,413.0,373.0,354.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [46]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 81)]
lag_features.append('rolling_avg')
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df3[lag_features]
y = df3['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 10905, number of used features: 83
[LightGBM] [Info] Start training from score -0.078944
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001719 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 21810, number of used features: 83
[LightGBM] [Info] Start training from score 0.003460


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002163 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 32715, number of used features: 83
[LightGBM] [Info] Start training from score 0.067810
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 43620, number of used features: 83
[LightGBM] [Info] Start training from score 0.028142


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003419 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 54525, number of used features: 83
[LightGBM] [Info] Start training from score -0.001925
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.758
MAE : 0.555
R2  : 42.60 %

----- Decision Tree -----
RMSE: 0.995
MAE : 0.735
R2  : 0.91 %

----- Random Forest -----
RMSE: 0.759
MAE : 0.558
R2  : 42.38 %

----- Gradient Boosting -----
RMSE: 0.757
MAE : 0.556
R2  : 42.63 %

----- Support Vector Regressor -----
RMSE: 0.774
MAE : 0.570
R2  : 40.02 %

----- XGBoost -----
RMSE: 0.784
MAE : 0.574
R2  : 38.51 %

----- LightGBM -----
RMSE: 0.759
MAE : 0.557
R2  : 42.38 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 4 : TimeSeriesSplit + lag_60 + rolling_avg_7**

In [47]:
# 2. Sort the dataframe by the full date (oldest to latest)
df4=df.copy()
df4 = df4.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df4['rolling_avg'] = df4['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 61):
    df4[f'lag{i}'] = df4['daily_columno3'].shift(i)


df4.dropna(inplace=True)
print(df4.shape)
df4.head()

(65450, 63)


,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [48]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
X = df4[lag_features]
y = df4['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000936 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 10910, number of used features: 61
[LightGBM] [Info] Start training from score -0.074399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001272 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 21818, number of used features: 61
[LightGBM] [Info] Start training from score 0.004540


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001779 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 32726, number of used features: 61
[LightGBM] [Info] Start training from score 0.068249
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002031 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 43634, number of used features: 61
[LightGBM] [Info] Start training from score 0.028112


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002777 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 54542, number of used features: 61
[LightGBM] [Info] Start training from score -0.001818
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.758
MAE : 0.555
R2  : 42.58 %

----- Decision Tree -----
RMSE: 1.004
MAE : 0.740
R2  : -0.83 %

----- Random Forest -----
RMSE: 0.758
MAE : 0.558
R2  : 42.47 %

----- Gradient Boosting -----
RMSE: 0.756
MAE : 0.555
R2  : 42.80 %

----- Support Vector Regressor -----
RMSE: 0.772
MAE : 0.568
R2  : 40.34 %

----- XGBoost -----
RMSE: 0.782
MAE : 0.572
R2  : 38.90 %

----- LightGBM -----
RMSE: 0.759
MAE : 0.557
R2  : 42.39 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 5 : TimeSeriesSplit + lag_60 + rolling_avg_7**

In [49]:
# 2. Sort the dataframe by the full date (oldest to latest)
df5=df.copy()
df5 = df5.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df5['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 61):
    df5[f'lag{i}'] = df5['daily_columno3'].shift(i)


df5.dropna(inplace=True)
print(df5.shape)
df5.head()

(65450, 63)


,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [50]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
X = df5[lag_features]
y = df5['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000768 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 10910, number of used features: 61
[LightGBM] [Info] Start training from score -0.074399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001040 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 21818, number of used features: 61
[LightGBM] [Info] Start training from score 0.004540


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001729 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 32726, number of used features: 61
[LightGBM] [Info] Start training from score 0.068249
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002302 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 43634, number of used features: 61
[LightGBM] [Info] Start training from score 0.028112


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002673 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 54542, number of used features: 61
[LightGBM] [Info] Start training from score -0.001818
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.758
MAE : 0.555
R2  : 42.58 %

----- Decision Tree -----
RMSE: 1.004
MAE : 0.740
R2  : -0.83 %

----- Random Forest -----
RMSE: 0.758
MAE : 0.558
R2  : 42.47 %

----- Gradient Boosting -----
RMSE: 0.756
MAE : 0.555
R2  : 42.80 %

----- Support Vector Regressor -----
RMSE: 0.772
MAE : 0.568
R2  : 40.34 %

----- XGBoost -----
RMSE: 0.782
MAE : 0.572
R2  : 38.90 %

----- LightGBM -----
RMSE: 0.759
MAE : 0.557
R2  : 42.39 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 6 : TimeSeriesSplit + lag_30 + exp_avg_3**

In [51]:
# 2. Sort the dataframe by the full date (oldest to latest)
df6=df.copy()
df6 = df6.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df6['ema_avg'] = df6['daily_columno3'].shift(1).ewm(span=3, adjust=False).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 31):
    df6[f'lag{i}'] = df6['daily_columno3'].shift(i)


df6.dropna(inplace=True)
print(df6.shape)
df6.head()

(65480, 33)


,daily_date,daily_columno3,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag21,lag22,lag23,lag24,lag25,lag26,lag27,lag28,lag29,lag30
30,1976-09-07,273.8,287.181151,289.4,280.6,284.5,290.3,295.2,303.0,300.1,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
31,1976-09-08,284.5,280.490575,273.8,289.4,280.6,284.5,290.3,295.2,303.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
32,1976-09-09,283.5,282.495288,284.5,273.8,289.4,280.6,284.5,290.3,295.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
33,1976-09-10,283.5,282.997644,283.5,284.5,273.8,289.4,280.6,284.5,290.3,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
34,1976-09-13,307.9,283.248822,283.5,283.5,284.5,273.8,289.4,280.6,284.5,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [52]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 31)]
lag_features.append('ema_avg')
X = df6[lag_features]
y = df6['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000366 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 10915, number of used features: 31
[LightGBM] [Info] Start training from score -0.074213
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 21828, number of used features: 31
[LightGBM] [Info] Start training from score 0.003913
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000817 seconds.
You can set `f

/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000970 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 43654, number of used features: 31
[LightGBM] [Info] Start training from score 0.027864
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001208 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 54567, number of used features: 31
[LightGBM] [Info] Start training from score -0.001854
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.758
MAE : 0.555
R2  : 42.55 %

----- Decision Tree -----
RMSE: 1.001
MAE : 0.736
R2  : -0.13 %

----- Random Forest -----
RMSE: 0.760
MAE : 0.558
R2  : 42.25 %

----- Gradient Boosting -----
RMSE: 0.759
MAE : 0.556
R2  : 42.35 %

----- Support Vector

/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 7 : TimeSeriesSplit + lag_60 + rolling_std_7**

In [53]:
# 2. Sort the dataframe by the full date (oldest to latest)
df7=df.copy()
df7 = df7.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df7['rolling_std'] = df7['daily_columno3'].shift(1).rolling(window=7).std()


for i in range(1, 61):
    df7[f'lag{i}'] = df7['daily_columno3'].shift(i)


df7.dropna(inplace=True)
print(df7.shape)
df7.head()

(65450, 63)


,daily_date,daily_columno3,rolling_std,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,13.403820,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,22.845746,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,18.766231,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,48.539017,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,48.328696,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [54]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
X = df7[lag_features]
y = df7['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 10910, number of used features: 61
[LightGBM] [Info] Start training from score -0.074399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 21818, number of used features: 61
[LightGBM] [Info] Start training from score 0.004540


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001721 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 32726, number of used features: 61
[LightGBM] [Info] Start training from score 0.068249
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002362 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 43634, number of used features: 61
[LightGBM] [Info] Start training from score 0.028112


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002656 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 54542, number of used features: 61
[LightGBM] [Info] Start training from score -0.001818
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.756
MAE : 0.554
R2  : 42.83 %

----- Decision Tree -----
RMSE: 0.998
MAE : 0.736
R2  : 0.47 %

----- Random Forest -----
RMSE: 0.759
MAE : 0.558
R2  : 42.41 %

----- Gradient Boosting -----
RMSE: 0.758
MAE : 0.555
R2  : 42.51 %

----- Support Vector Regressor -----
RMSE: 0.772
MAE : 0.567
R2  : 40.33 %

----- XGBoost -----
RMSE: 0.783
MAE : 0.573
R2  : 38.62 %

----- LightGBM -----
RMSE: 0.760
MAE : 0.556
R2  : 42.24 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Trial 8 : TimeSeriesSplit + lag_60 + rolling_avg_3 + rolling_std_3***

In [55]:
# 2. Sort the dataframe by the full date (oldest to latest)
df8=df.copy()
df8 = df8.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df8['rolling_std'] = df8['daily_columno3'].shift(1).rolling(window=3).std()
df8['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=3).mean()


for i in range(1, 61):
    df8[f'lag{i}'] = df8['daily_columno3'].shift(i)


df8.dropna(inplace=True)
print(df8.shape)
df8.head()

(65447, 64)


,daily_date,daily_columno3,rolling_std,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
63,1980-03-06,323.0,59.911045,357.333333,424.0,308.0,340.0,286.0,287.4,297.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,63.089883,351.666667,323.0,424.0,308.0,340.0,286.0,287.4,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7
65,1980-03-13,363.0,50.500825,373.333333,373.0,323.0,424.0,308.0,340.0,286.0,...,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8
66,1980-03-24,381.0,26.457513,353.000000,363.0,373.0,323.0,424.0,308.0,340.0,...,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8
67,1980-03-25,333.0,9.018500,372.333333,381.0,363.0,373.0,323.0,424.0,308.0,...,307.9,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3


In [56]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
lag_features.append('rolling_avg')
X = df8[lag_features]
y = df8['daily_columno3']


X_scaled = minmax_x.fit_transform(X)
y_scaled = minmax_y.fit_transform(y.values.reshape(-1, 1)).flatten()


# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}
tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    fold_scores = []
    preds_all = np.zeros_like(y_scaled)
    for train_index, test_index in tscv.split(X_scaled):
        X_train, X_test = X_scaled[train_index], X_scaled[test_index]
        y_train, y_test = y_scaled[train_index], y_scaled[test_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds_all[test_index] = preds

    mse = mean_squared_error(y_scaled, preds_all)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_scaled, preds_all)
    r2 = r2_score(y_scaled, preds_all)

    results[name] = {
        'Model': model,
        'Predictions': preds_all,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



Training Linear Regression...

Training Decision Tree...

Training Random Forest...

Training Gradient Boosting...

Training Support Vector Regressor...

Training XGBoost...

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 10912, number of used features: 62
[LightGBM] [Info] Start training from score -0.075370
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001186 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 21819, number of used features: 62
[LightGBM] [Info] Start training from score 0.004317


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001765 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 32726, number of used features: 62
[LightGBM] [Info] Start training from score 0.068191
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002077 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 43633, number of used features: 62
[LightGBM] [Info] Start training from score 0.028150


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002223 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 54540, number of used features: 62
[LightGBM] [Info] Start training from score -0.001840
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.755
MAE : 0.553
R2  : 43.02 %

----- Decision Tree -----
RMSE: 1.003
MAE : 0.738
R2  : -0.67 %

----- Random Forest -----
RMSE: 0.760
MAE : 0.558
R2  : 42.30 %

----- Gradient Boosting -----
RMSE: 0.758
MAE : 0.555
R2  : 42.54 %

----- Support Vector Regressor -----
RMSE: 0.772
MAE : 0.566
R2  : 40.39 %

----- XGBoost -----
RMSE: 0.785
MAE : 0.573
R2  : 38.45 %

----- LightGBM -----
RMSE: 0.761
MAE : 0.557
R2  : 42.16 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
